In [2]:
# ============================================================
# CELL 1 — INSTALLS
# ============================================================

!pip install -q torch torchvision qai-hub onnx onnxruntime

import json
import os
import numpy as np
import torch
import torch.nn as nn
import torchvision.models.video as video_models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")
print(f"   PyTorch: {torch.__version__}")

✅ Device: cuda
   PyTorch: 2.10.0+cu128


In [3]:
# ============================================================
# CONFIG (portable across Kaggle runs)
# ============================================================

import shutil

OUTPUT_DIR = os.getenv("OUTPUT_DIR", "/kaggle/working")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Training artifacts (your provided input tree) ---
TRAINING_ROOT = os.getenv(
    "TRAINING_ROOT",
    "/kaggle/input/notebooks/dadhichisarkershayon/qevd-trainning-part-2",
)
TRAINING_DIR_CANDIDATES = [
    TRAINING_ROOT,
    os.path.join(TRAINING_ROOT, "QEVD-Trainning-Part-2"),
    "/kaggle/input/qevd-trainning-part-2",
    "/kaggle/input/qevd-training-part-2",
]
TRAINING_DIR = next((p for p in TRAINING_DIR_CANDIDATES if os.path.isdir(p)), "")

CKPT_PATH_ENV = os.getenv("CKPT_PATH", "")
CKPT_CANDIDATES = [
    CKPT_PATH_ENV,
    os.path.join(TRAINING_DIR, "best_r2plus1d_qevd.pth") if TRAINING_DIR else "",
    os.path.join(TRAINING_DIR, "latest_checkpoint.pth") if TRAINING_DIR else "",
    "/kaggle/working/best_r2plus1d_qevd.pth",
]
CKPT_PATH = next((p for p in CKPT_CANDIDATES if p and os.path.isfile(p)), "")

PREBUILT_ONNX_ENV = os.getenv("PREBUILT_ONNX_PATH", "")
PREBUILT_ONNX_CANDIDATES = [
    PREBUILT_ONNX_ENV,
    os.path.join(TRAINING_DIR, "lpcvc_final_unified.onnx") if TRAINING_DIR else "",
]
PREBUILT_ONNX_PATH = next((p for p in PREBUILT_ONNX_CANDIDATES if p and os.path.isfile(p)), "")

# --- Preprocessing artifacts (your provided input tree) ---
PREPROCESS_ROOT = os.getenv(
    "PREPROCESS_ROOT",
    "/kaggle/input/notebooks/dadhichisarkershayon/qevd-preprocessing-part-2",
)
PREPROCESS_DIR_CANDIDATES = [
    PREPROCESS_ROOT,
    os.path.join(PREPROCESS_ROOT, "QEVD-Preprocessing-Part-2"),
    "/kaggle/input/qevd-preprocessed",
]
PREPROCESS_DIR = next((p for p in PREPROCESS_DIR_CANDIDATES if os.path.isdir(p)), "")

MAPPING_PATH_ENV = os.getenv("MAPPING_PATH", "")
MAPPING_CANDIDATES = [
    MAPPING_PATH_ENV,
    os.path.join(PREPROCESS_DIR, "preprocessed_tensors", "class_map.json") if PREPROCESS_DIR else "",
    "/kaggle/input/notebooks/dadhichisarkershayon/qevd-preprocessing-part-2/preprocessed_tensors/class_map.json",
    "/kaggle/input/qevd-preprocessed/preprocessed_tensors/class_map.json",
    "/kaggle/input/qevd-preprocessed/class_map.json",
]
MAPPING_PATH = next((p for p in MAPPING_CANDIDATES if p and os.path.isfile(p)), "")

# Unified ONNX path used by compile/profile cells
ONNX_PATH = os.path.join(OUTPUT_DIR, "lpcvc_final_unified.onnx")

# Competition constants
NUM_FRAMES = 16
FRAME_SIZE = 112
LATENCY_LIMIT_MS = 34.0
AIHUB_DEVICE = "Dragonwing IQ-9075 EVK"

if not MAPPING_PATH:
    raise FileNotFoundError(
        "Mapping file not found. Set MAPPING_PATH or attach QEVD-Preprocessing-Part-2 input."
    )

if not CKPT_PATH and not PREBUILT_ONNX_PATH:
    raise FileNotFoundError(
        "No checkpoint or prebuilt ONNX found. Attach QEVD-Trainning-Part-2 input."
    )

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    mapping = json.load(f)

if isinstance(mapping, dict) and all(isinstance(v, int) for v in mapping.values()):
    NUM_CLASSES = len(mapping)
    CLASSES = [k for k, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
else:
    raise ValueError("Expected class_map.json format {class_name: index}")

print("Paths configured")
print(f"Preprocess dir: {PREPROCESS_DIR or 'not found'}")
print(f"Training dir:   {TRAINING_DIR or 'not found'}")
print(f"Checkpoint:     {CKPT_PATH or 'not found'}")
print(f"Prebuilt ONNX:  {PREBUILT_ONNX_PATH or 'not found'}")
print(f"Mapping:        {MAPPING_PATH}")
print(f"Classes:        {NUM_CLASSES}")
print(f"ONNX out:       {ONNX_PATH}")

Paths configured
Preprocess dir: /kaggle/input/notebooks/dadhichisarkershayon/qevd-preprocessing-part-2
Training dir:   /kaggle/input/notebooks/dadhichisarkershayon/qevd-trainning-part-2
Checkpoint:     /kaggle/input/notebooks/dadhichisarkershayon/qevd-trainning-part-2/best_r2plus1d_qevd.pth
Prebuilt ONNX:  /kaggle/input/notebooks/dadhichisarkershayon/qevd-trainning-part-2/lpcvc_final_unified.onnx
Mapping:        /kaggle/input/notebooks/dadhichisarkershayon/qevd-preprocessing-part-2/preprocessed_tensors/class_map.json
Classes:        92
ONNX out:       /kaggle/working/lpcvc_final_unified.onnx


In [4]:
# ============================================================
# CELL 4 — REBUILD MODEL & LOAD WEIGHTS
# ============================================================

def build_model(num_classes, backbone_name):
    if backbone_name == "r2plus1d_18":
        model = video_models.r2plus1d_18(weights=None)
    elif backbone_name == "mc3_18":
        model = video_models.mc3_18(weights=None)
    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    in_feat  = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_feat, num_classes),
    )
    return model

# Load checkpoint
# Resolve checkpoint path (handles relative path issues)
ckpt_candidates = [
    CKPT_PATH,
    os.path.join(OUTPUT_DIR, os.path.basename(CKPT_PATH)),
    os.path.join(os.path.dirname(MAPPING_PATH), os.path.basename(CKPT_PATH)),
]
ckpt_file = next((p for p in ckpt_candidates if os.path.isfile(p)), None)

if ckpt_file is None:
    raise FileNotFoundError(
        f"Checkpoint not found. Tried: {ckpt_candidates}\n"
        f"Current working dir: {os.getcwd()}"
    )

# Load checkpoint (compatible with different torch versions / checkpoint formats)
try:
    raw_ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=True)
except TypeError:
    raw_ckpt = torch.load(ckpt_file, map_location="cpu")

# Extract the actual state_dict from common checkpoint formats
if isinstance(raw_ckpt, dict) and "model_state" in raw_ckpt and isinstance(raw_ckpt["model_state"], dict):
    state_dict = raw_ckpt["model_state"]
elif isinstance(raw_ckpt, dict) and "state_dict" in raw_ckpt and isinstance(raw_ckpt["state_dict"], dict):
    state_dict = raw_ckpt["state_dict"]
elif isinstance(raw_ckpt, dict) and "model" in raw_ckpt and isinstance(raw_ckpt["model"], dict):
    state_dict = raw_ckpt["model"]
elif isinstance(raw_ckpt, dict):
    state_dict = raw_ckpt
else:
    raise ValueError(f"Unsupported checkpoint format in: {ckpt_file}")

# Remove DataParallel prefix if present
state_dict = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in state_dict.items()}

ckpt = {
    "model_state": state_dict,
    "epoch": raw_ckpt.get("epoch", -1) if isinstance(raw_ckpt, dict) else -1,
    "val_acc": raw_ckpt.get("val_acc", float("nan")) if isinstance(raw_ckpt, dict) else float("nan"),
}

# Pick backbone from checkpoint file name (your file indicates r2plus1d)
ckpt_name = os.path.basename(ckpt_file).lower()
backbone_name = "r2plus1d_18" if "r2plus1d" in ckpt_name else "mc3_18"

print(f"✅ Using checkpoint: {ckpt_file}")
print(f"✅ Backbone selected: {backbone_name}")

model = build_model(NUM_CLASSES, backbone_name)
# Handle checkpoint heads saved as a plain Linear layer
if "fc.weight" in ckpt["model_state"] and "fc.1.weight" not in ckpt["model_state"]:
    ckpt["model_state"]["fc.1.weight"] = ckpt["model_state"].pop("fc.weight")
    ckpt["model_state"]["fc.1.bias"]   = ckpt["model_state"].pop("fc.bias")

model.load_state_dict(ckpt["model_state"], strict=True)
model.eval()

print(f"✅ Model loaded")
print(f"   Trained epochs : {ckpt['epoch']}")
print(f"   Best val acc   : {ckpt['val_acc']:.2f}%")
print(f"   Num classes    : {NUM_CLASSES}")

# Sanity check output shape
with torch.no_grad():
    dummy = torch.randn(1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE)
    out   = model(dummy)
    print(f"   Output shape   : {out.shape}  ✓")
    del dummy, out

✅ Using checkpoint: /kaggle/input/notebooks/dadhichisarkershayon/qevd-trainning-part-2/best_r2plus1d_qevd.pth
✅ Backbone selected: r2plus1d_18
✅ Model loaded
   Trained epochs : 24
   Best val acc   : 87.71%
   Num classes    : 92
   Output shape   : torch.Size([1, 92])  ✓


In [5]:
!pip install -q onnx onnxruntime onnxscript qai-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.4 MB/s eta 0:00:00


In [6]:
# ============================================================
# CELL 5 — RESOLVE/EXPORT TO ONNX (supports prebuilt ONNX)
# ============================================================

import onnx
import onnxruntime as ort

if PREBUILT_ONNX_PATH and os.path.isfile(PREBUILT_ONNX_PATH):
    print(f"📦 Using prebuilt ONNX: {PREBUILT_ONNX_PATH}")
    if os.path.abspath(PREBUILT_ONNX_PATH) != os.path.abspath(ONNX_PATH):
        shutil.copy2(PREBUILT_ONNX_PATH, ONNX_PATH)
    print(f"✅ ONNX ready at: {ONNX_PATH}")
else:
    if "model" not in globals():
        raise RuntimeError(
            "Model not loaded. Run the model build/load cell first, or provide PREBUILT_ONNX_PATH."
        )

    print("📦 Exporting to ONNX ...")
    dummy = torch.randn(1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE)

    ONNX_TMP = os.path.join(OUTPUT_DIR, "lpcvc_final_unified_tmp.onnx")

    # Use legacy exporter to avoid requiring onnxscript
    torch.onnx.export(
        model,
        dummy,
        ONNX_TMP,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes=None,
        verbose=False,
        dynamo=False,
    )
    print("✅ Initial export done")

    print("🔧 Inlining external weights ...")
    onnx_model = onnx.load(ONNX_TMP, load_external_data=True)
    onnx.save(
        onnx_model,
        ONNX_PATH,
        save_as_external_data=False,
    )

size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f"✅ ONNX saved → {ONNX_PATH}  ({size_mb:.2f} MB)")

onnx.checker.check_model(ONNX_PATH)
print("✅ ONNX checker passed")

# Parity check only if model exists in memory
if "model" in globals():
    dummy = torch.randn(1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE)
    with torch.no_grad():
        torch_out = model(dummy).numpy()
    sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
    ort_out = sess.run(None, {"input": dummy.numpy()})[0]
    max_diff = float(np.abs(torch_out - ort_out).max())
    print(f"✅ Max PyTorch ↔ ONNX diff : {max_diff:.6f}  "
          f"{'✓ OK' if max_diff < 1e-3 else '⚠ check this'}")
else:
    print("ℹ️ Parity check skipped (no model loaded).")

📦 Using prebuilt ONNX: /kaggle/input/notebooks/dadhichisarkershayon/qevd-trainning-part-2/lpcvc_final_unified.onnx
✅ ONNX ready at: /kaggle/working/lpcvc_final_unified.onnx
✅ ONNX saved → /kaggle/working/lpcvc_final_unified.onnx  (119.55 MB)
✅ ONNX checker passed
✅ Max PyTorch ↔ ONNX diff : 0.000002  ✓ OK


In [7]:
# Quick version diagnostic — run this alone first
import qai_hub
print(f"qai_hub version : {qai_hub.__version__}")
print(f"Available attrs : {[x for x in dir(qai_hub) if not x.startswith('_')]}")

qai_hub version : 0.47.0
Available attrs : ['Client', 'ClientConfig', 'CompileJob', 'CompileJobResult', 'CompileJobSummary', 'Dataset', 'Device', 'Error', 'Framework', 'InferenceJob', 'InferenceJobResult', 'InferenceJobSummary', 'InputSpecs', 'InternalError', 'Job', 'JobArtifactType', 'JobResult', 'JobStatus', 'JobSummary', 'JobType', 'LinkJob', 'LinkJobResult', 'LinkJobSummary', 'Model', 'ModelMetadataKey', 'ProfileJob', 'ProfileJobResult', 'ProfileJobSummary', 'Project', 'QuantizeDtype', 'QuantizeJob', 'QuantizeJobResult', 'QuantizeJobSummary', 'SourceModelType', 'UserError', 'api_status_codes', 'api_utils', 'client', 'get_dataset', 'get_datasets', 'get_device_attributes', 'get_devices', 'get_frameworks', 'get_job', 'get_job_summaries', 'get_model', 'get_models', 'hub', 'public_api_pb2', 'public_rest_api', 'set_session_token', 'set_verbose', 'submit_compile_and_link_jobs', 'submit_compile_and_profile_jobs', 'submit_compile_job', 'submit_inference_job', 'submit_link_job', 'submit_prof

### Qualcomm AI Hub authentication (Kaggle)

If you see `/root/.qai_hub/client.ini not found`, use token auth instead of local config.

1. In Kaggle, go to **Add-ons -> Secrets**.
2. Add a secret named `QAI_HUB_TOKEN` (or `QAIHUB_TOKEN`).
3. Run the next diagnostic cell again.

The notebook now loads token from environment or Kaggle Secrets automatically.

In [16]:
!qai-hub configure --api_token wyfb5d78py0bgow6gz10krdqyd8qq0l9wlfw9swc

2026-04-11 10:11:18.606 - INFO - Enabling verbose logging.
/usr/local/lib/python3.12/dist-packages/qai_hub/_cli.py:412: UserWarning: Overwriting configuration: /root/.qai_hub/client.ini (previous configuration saved to /root/.qai_hub/client.ini.bak)
  warnings.warn(
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = wyfb5d78py0bgow6gz10krdqyd8qq0l9wlfw9swc
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




In [17]:
!qai-hub list-devices

+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|              Device             |      OS      |  Vendor  |   Type  |                      Chipset                      |                       CLI Invocation                       |
+---------------------------------+--------------+----------+---------+---------------------------------------------------+------------------------------------------------------------+
|     Google Pixel 3 (Family)     |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |     --device "Google Pixel 3 (Family)" --device-os 10      |
|          Google Pixel 3         |  Android 10  |  Google  |  Phone  |          qualcomm-snapdragon-845, sdm845          |          --device "Google Pixel 3" --device-os 10          |
|         Google Pixel 3a         |  Android 10  |  Google  |  Phone  |    

In [18]:
# ============================================================
# COMPILE ON QUALCOMM AI HUB (sharing moved to final cell)
# ============================================================

import importlib
import qai_hub
importlib.reload(qai_hub)

try:
    from kaggle_secrets import UserSecretsClient
except ModuleNotFoundError:
    UserSecretsClient = None

def _get_qai_hub_token():
    # 1) Environment variables
    for key in ("QAI_HUB_TOKEN", "QAIHUB_TOKEN"):
        val = os.getenv(key)
        if val and val.strip():
            return val.strip()

    # 2) Kaggle Secrets
    if UserSecretsClient is not None:
        client = UserSecretsClient()
        for key in ("QAI_HUB_TOKEN", "QAIHUB_TOKEN"):
            try:
                val = client.get_secret(key)
                if val and str(val).strip():
                    return str(val).strip()
            except Exception:
                pass

    # 3) Interactive fallback (for local Jupyter)
    import getpass
    val = getpass.getpass("Enter Qualcomm AI Hub token: ").strip()
    if val:
        return val

    raise RuntimeError(
        "Missing Qualcomm AI Hub token. Set QAI_HUB_TOKEN/QAIHUB_TOKEN or add a Kaggle secret."
    )

token = _get_qai_hub_token()
os.environ["QAI_HUB_TOKEN"] = token
hub = qai_hub.Client()
print("Authenticated with Qualcomm AI Hub")

if not os.path.isfile(ONNX_PATH):
    raise FileNotFoundError(f"ONNX not found at {ONNX_PATH}. Run export cell first.")

devices = hub.get_devices(name=AIHUB_DEVICE)
if not devices:
    raise RuntimeError(
        f"Device '{AIHUB_DEVICE}' not available for this account right now. "
        "Run the device diagnostic cell and pick an available Dragonwing target."
    )
device = devices[0]
print(f"Target device: {device.name}")

print("Uploading ONNX model ...")
hub_model = hub.upload_model(ONNX_PATH)
print(f"Uploaded model: {hub_model}")

print("Compiling ...")
compile_job = hub.submit_compile_job(
    model=hub_model,
    device=device,
    name="LPCVC2026_Track2_R2Plus1D18",
    options="--target_runtime qnn_context_binary",
    input_specs={"input": ((1, 3, NUM_FRAMES, FRAME_SIZE, FRAME_SIZE), "float32")},
)
COMPILE_JOB_ID = compile_job.job_id
print(f"Compile job: {COMPILE_JOB_ID}")
compile_job.wait()

status = compile_job.get_status()
if not status.success:
    raise RuntimeError(f"Compile failed: {status.message}")

print("Compilation succeeded")
target_model = compile_job.get_target_model()
print("Sharing is deferred. Run the final cell when you are ready to share the compile job.")

Authenticated with Qualcomm AI Hub
Target device: Dragonwing IQ-9075 EVK
Uploading ONNX model ...
Uploading lpcvc_final_unified.onnx


100%|██████████| 120M/120M [00:01<00:00, 84.8MB/s] 


Uploaded model: Model(model_id='mnj7133dn', name='lpcvc_final_unified.onnx')
Compiling ...
Scheduled compile job (jpx7xwd9g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpx7xwd9g/

Compile job: jpx7xwd9g
Waiting for compile job (jpx7xwd9g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Compilation succeeded
Sharing is deferred. Run the final cell when you are ready to share the compile job.


In [19]:
# ============================================================
# PROFILE LATENCY ON DEVICE
# ============================================================

import json, glob

print("Profiling on device ...")
profile_job = hub.submit_profile_job(
    model=target_model,
    device=device,
    name="LPCVC2026_Track2_R2Plus1D18_profile",
)
print(f"Profile job: {profile_job.job_id}")
profile_job.wait()

profile_dir = os.path.join(OUTPUT_DIR, "profile_results")
os.makedirs(profile_dir, exist_ok=True)
profile_job.download_results(artifacts_dir=profile_dir)

json_files = glob.glob(os.path.join(profile_dir, "*.json"))
if not json_files:
    raise RuntimeError("No profile JSON found after profiling.")

with open(sorted(json_files)[-1]) as f:
    profile_data = json.load(f)

# AI Hub profile reports estimated_inference_time in microseconds
inf_time_us = profile_data["execution_summary"]["estimated_inference_time"]
inf_time_ms = inf_time_us / 1000.0
all_times_ms = [t / 1000.0 for t in profile_data["execution_summary"]["all_inference_times"]]
peak_mem_mb = profile_data["execution_summary"]["estimated_inference_peak_memory"] / 1024 / 1024

print("=" * 60)
print(f"Inference time (best): {inf_time_ms:.2f} ms")
print(f"Inference time (mean): {sum(all_times_ms)/len(all_times_ms):.2f} ms")
print(f"Inference time (max):  {max(all_times_ms):.2f} ms")
print(f"Peak memory:           {peak_mem_mb:.2f} MB")
print(f"Latency limit:         {LATENCY_LIMIT_MS:.2f} ms")
if inf_time_ms < LATENCY_LIMIT_MS:
    print(f"VALID: margin = {LATENCY_LIMIT_MS - inf_time_ms:.2f} ms")
else:
    print(f"OVER LIMIT by {inf_time_ms - LATENCY_LIMIT_MS:.2f} ms")
print("=" * 60)

print("Next: submit the compile job ID in the LPCVC form.")

Profiling on device ...
Scheduled profile job (jp016nmeg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp016nmeg/

Profile job: jp016nmeg
Waiting for profile job (jp016nmeg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


jp016nmeg_runtime.log: 100%|██████████| 35.7k/35.7k [00:00<00:00, 17.0MB/s]

Saved profile results to /kaggle/working/profile_results/LPCVC2026_Track2_R2Plus1D18_profile_jp016nmeg_results.json
Inference time (best): 27.48 ms
Inference time (mean): 28.73 ms
Inference time (max):  29.28 ms
Peak memory:           8.61 MB
Latency limit:         34.00 ms
VALID: margin = 6.52 ms
Next: submit the compile job ID in the LPCVC form.


In [20]:
# ============================================================
# FINAL STEP — SHARE COMPILE JOB (RUN ONLY WHEN READY)
# ============================================================

SHARE_EMAIL = "lowpowervision@gmail.com"

if "compile_job" not in globals():
    if "COMPILE_JOB_ID" not in globals():
        raise RuntimeError(
            "compile_job not found in memory. Run the compile cell first so COMPILE_JOB_ID is available."
        )
    if "hub" not in globals():
        import qai_hub
        hub = qai_hub.Client()
    compile_job = hub.get_job(COMPILE_JOB_ID)

compile_job.modify_sharing(add_emails=[SHARE_EMAIL])
print(f"✅ Compile job shared with {SHARE_EMAIL}")
print(f"✅ Compile job ID: {compile_job.job_id}")

✅ Compile job shared with lowpowervision@gmail.com
✅ Compile job ID: jpx7xwd9g
